# 03 - Integracao e Feature Engineering

Objetivo deste notebook:
- integrar as bases tratadas por `codigo_municipio + ano`
- criar a base analitica principal do projeto
- gerar features para BI, Data Warehouse e Machine Learning
- salvar a base final em `datasets/tratados/base_analitica_municipio_ano.csv`

Granularidade da base final:

`1 linha = 1 capital brasileira + 1 ano`


## 1. Imports e caminhos

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PASTA_TRATADOS = Path('../datasets/tratados')
CAMINHO_BASE_ANALITICA = PASTA_TRATADOS / 'base_analitica_municipio_ano.csv'


## 2. Funcoes auxiliares

In [2]:
def validar_chave(df: pd.DataFrame, chave: list[str], nome: str) -> None:
    duplicatas = df.duplicated(subset=chave).sum()
    print(f'{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas')
    print(f'Duplicatas na chave {chave}: {duplicatas}')
    if duplicatas:
        display(df[df.duplicated(subset=chave, keep=False)].sort_values(chave).head(20))


def adicionar_taxa_100k(df: pd.DataFrame, coluna_numerador: str, coluna_saida: str) -> pd.DataFrame:
    resultado = df.copy()
    resultado[coluna_saida] = (resultado[coluna_numerador] / resultado['populacao_total']) * 100000
    return resultado


def mostrar_nulos(df: pd.DataFrame) -> None:
    display(df.isna().sum().sort_values(ascending=False).to_frame('qtd_nulos'))


## 3. Carregar bases tratadas

In [3]:
df_idh = pd.read_csv(PASTA_TRATADOS / 'idh_municipio_2010_tratado.csv')
df_crimes = pd.read_csv(PASTA_TRATADOS / 'crimes_municipio_ano_tratado.csv')
df_populacao = pd.read_csv(PASTA_TRATADOS / 'populacao_municipio_ano_tratado.csv')
df_educacao = pd.read_csv(PASTA_TRATADOS / 'educacao_ideb_uf_ano_total_em_tratado.csv')

bases = {
    'idh': df_idh,
    'crimes': df_crimes,
    'populacao': df_populacao,
    'educacao': df_educacao,
}

for nome, df in bases.items():
    print(nome, df.shape)
    display(df.head())


idh (27, 11)


,codigo_municipio,codigo_uf_ibge,sigla_uf,nome_municipio,nome_municipio_padronizado,ano_referencia_idhm,idhm,idhm_renda,idhm_educacao,idhm_longevidade,territorialidade
0,2800308,28,SE,Aracaju,aracaju,2010,0.770,0.784,0.708,0.823,Aracaju (SE)
1,1501402,15,PA,Belém,belem,2010,0.746,0.751,0.673,0.822,Belém (PA)
2,3106200,31,MG,Belo Horizonte,belo horizonte,2010,0.810,0.841,0.737,0.856,Belo Horizonte (MG)
3,1400100,14,RR,Boa Vista,boa vista,2010,0.752,0.737,0.708,0.816,Boa Vista (RR)
4,5300108,53,DF,Brasília,brasilia,2010,0.824,0.863,0.742,0.873,Brasília (DF)


crimes (162, 31)


,codigo_municipio,sigla_uf,nome_municipio,nome_municipio_padronizado,ano,quantidade_mortes_intervencao_policial_civil_fora_de_servico,quantidade_feminicidio,quantidade_mortes_intervencao_policial_militar_fora_de_servico,quantidade_furto_veiculos,quantidade_mortes_intervencao_policial_civil_em_servico,...,quantidade_roubo_furto_veiculos,quantidade_posse_ilegal_arma_de_fogo,quantidade_lesao_corporal_dolosa_violencia_domestica,quantidade_trafico_entorpecente,quantidade_roubo_veiculos,quantidade_lesao_corporal_morte,quantidade_morte_policiais_militares_confronto_em_servico,quantidade_posse_ilegal_porte_ilegal_arma_de_fogo,quantidade_homicidio_doloso,proporcao_mortes_intenvencao_policial_x_mortes_violentas_intencionais
0,2704302,AL,Maceió,maceio,2016,0.0,0.0,2.0,305.0,0.0,...,1538.0,0.0,0.0,0.0,1233.0,2.0,2.0,0.0,449,11.0
1,2304400,CE,Fortaleza,fortaleza,2016,0.0,0.0,7.0,2820.0,1.0,...,9235.0,0.0,0.0,0.0,6415.0,15.0,1.0,0.0,965,3.0
2,3205309,ES,Vitória,vitoria,2016,0.0,0.0,2.0,338.0,1.0,...,565.0,0.0,0.0,0.0,227.0,3.0,0.0,0.0,51,14.0
3,5208707,GO,Goiânia,goiania,2016,0.0,0.0,24.0,3733.0,0.0,...,11031.0,0.0,0.0,0.0,7298.0,14.0,0.0,0.0,452,16.0
4,2111300,MA,São Luís,sao luis,2016,0.0,0.0,0.0,487.0,0.0,...,2165.0,0.0,0.0,0.0,1678.0,12.0,0.0,0.0,498,4.0


populacao (162, 8)


,codigo_municipio,codigo_uf_ibge,sigla_uf,nome_municipio,nome_municipio_padronizado,ano,populacao_total,populacao_crescimento_pct
0,1100205,11,RO,Porto Velho,porto velho,2016,499293,NaN
1,1100205,11,RO,Porto Velho,porto velho,2017,509323,2.008841
2,1100205,11,RO,Porto Velho,porto velho,2018,519531,2.004229
3,1100205,11,RO,Porto Velho,porto velho,2019,529544,1.927315
4,1100205,11,RO,Porto Velho,porto velho,2020,539354,1.852537


educacao (81, 8)


,codigo_uf_ibge,sigla_uf,ano,ideb,fluxo,aprendizado,nota_mt,nota_lp
0,11,RO,2017,4.0,0.8735,4.5271,272.03,268.33
1,12,AC,2017,3.8,0.8770,4.3461,263.28,264.45
2,13,AM,2017,3.5,0.8745,4.0231,250.93,254.46
3,14,RR,2017,3.5,0.8457,4.1174,256.73,255.32
4,15,PA,2017,3.1,0.8035,3.8280,246.29,245.78


## 4. Validar granularidade das entradas

In [4]:
validar_chave(df_idh, ['codigo_municipio'], 'IDH')
validar_chave(df_crimes, ['codigo_municipio', 'ano'], 'Crimes')
validar_chave(df_populacao, ['codigo_municipio', 'ano'], 'Populacao')
validar_chave(df_educacao, ['codigo_uf_ibge', 'ano'], 'Educacao IDEB UF ano')


IDH: 27 linhas, 11 colunas
Duplicatas na chave ['codigo_municipio']: 0
Crimes: 162 linhas, 31 colunas
Duplicatas na chave ['codigo_municipio', 'ano']: 0
Populacao: 162 linhas, 8 colunas
Duplicatas na chave ['codigo_municipio', 'ano']: 0
Educacao IDEB UF ano: 81 linhas, 8 colunas
Duplicatas na chave ['codigo_uf_ibge', 'ano']: 0


## 5. Integrar bases

In [5]:
df_base = df_populacao.copy()

colunas_crimes_para_remover = [
    'sigla_uf',
    'nome_municipio',
    'nome_municipio_padronizado',
]
df_crimes_metricas = df_crimes.drop(columns=[coluna for coluna in colunas_crimes_para_remover if coluna in df_crimes.columns])

df_base = df_base.merge(
    df_crimes_metricas,
    how='left',
    on=['codigo_municipio', 'ano'],
    validate='one_to_one'
)

df_idh_metricas = df_idh[[
    'codigo_municipio',
    'ano_referencia_idhm',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
]].copy()

df_base = df_base.merge(
    df_idh_metricas,
    how='left',
    on='codigo_municipio',
    validate='many_to_one'
)

df_educacao_metricas = df_educacao[[
    'codigo_uf_ibge',
    'ano',
    'ideb',
    'fluxo',
    'aprendizado',
    'nota_mt',
    'nota_lp',
]].copy()

df_base = df_base.merge(
    df_educacao_metricas,
    how='left',
    on=['codigo_uf_ibge', 'ano'],
    validate='many_to_one'
)

df_base.shape


(162, 44)

## 6. Tratar nulos apos integracao

In [6]:
mostrar_nulos(df_base)


,qtd_nulos
nota_lp,81
nota_mt,81
aprendizado,81
fluxo,81
ideb,81
populacao_crescimento_pct,27
proporcao_mortes_intenvencao_policial_x_mortes_violentas_intencionais,8
quantidade_posse_ilegal_porte_ilegal_arma_de_fogo,0
quantidade_posse_ilegal_arma_de_fogo,0
quantidade_lesao_corporal_dolosa_violencia_domestica,0


In [7]:
colunas_quantidade = [coluna for coluna in df_base.columns if coluna.startswith('quantidade_')]
colunas_proporcao = [coluna for coluna in df_base.columns if coluna.startswith('proporcao_')]

df_base[colunas_quantidade] = df_base[colunas_quantidade].fillna(0)
df_base[colunas_proporcao] = df_base[colunas_proporcao].fillna(0)

mostrar_nulos(df_base)


,qtd_nulos
nota_lp,81
nota_mt,81
aprendizado,81
fluxo,81
ideb,81
populacao_crescimento_pct,27
quantidade_posse_ilegal_porte_ilegal_arma_de_fogo,0
quantidade_posse_ilegal_arma_de_fogo,0
quantidade_lesao_corporal_dolosa_violencia_domestica,0
quantidade_trafico_entorpecente,0


## 7. Criar indicadores criminais agregados

In [8]:
componentes_crime_total = [
    'quantidade_mortes_violentas_intencionais',
    'quantidade_feminicidio',
    'quantidade_estupro',
    'quantidade_furto_veiculos',
    'quantidade_roubo_veiculos',
    'quantidade_latrocinio',
    'quantidade_lesao_corporal_morte',
]
componentes_crime_total = [coluna for coluna in componentes_crime_total if coluna in df_base.columns]

df_base['crimes_total_indicadores'] = df_base[componentes_crime_total].sum(axis=1)

df_base['homicidios_dolosos'] = df_base.get('quantidade_homicidio_doloso', 0)
df_base['mortes_violentas_intencionais'] = df_base.get('quantidade_mortes_violentas_intencionais', 0)
df_base['feminicidios'] = df_base.get('quantidade_feminicidio', 0)
df_base['estupros'] = df_base.get('quantidade_estupro', 0)
df_base['furto_veiculos'] = df_base.get('quantidade_furto_veiculos', 0)

df_base[[
    'codigo_municipio',
    'ano',
    'crimes_total_indicadores',
    'mortes_violentas_intencionais',
    'homicidios_dolosos',
    'feminicidios',
    'estupros',
    'furto_veiculos',
]].head()


,codigo_municipio,ano,crimes_total_indicadores,mortes_violentas_intencionais,homicidios_dolosos,feminicidios,estupros,furto_veiculos
0,1100205,2016,3882.0,191,173,0.0,382.0,1867.0
1,1100205,2017,3857.0,158,152,0.0,355.0,1860.0
2,1100205,2018,3863.0,160,152,4.0,355.0,1860.0
3,1100205,2019,4406.0,119,105,1.0,345.0,1991.0
4,1100205,2020,468.0,161,141,5.0,285.0,0.0


## 8. Criar taxas por 100 mil habitantes

In [9]:
taxas = {
    'crimes_total_indicadores': 'taxa_crimes_100k',
    'mortes_violentas_intencionais': 'taxa_mortes_violentas_100k',
    'homicidios_dolosos': 'taxa_homicidios_100k',
    'feminicidios': 'taxa_feminicidios_100k',
    'estupros': 'taxa_estupros_100k',
    'furto_veiculos': 'taxa_furto_veiculos_100k',
}

for numerador, saida in taxas.items():
    df_base = adicionar_taxa_100k(df_base, numerador, saida)

df_base[list(taxas.values())].describe().transpose()


,count,mean,std,min,25%,50%,75%,max
taxa_crimes_100k,162.0,357.891603,171.931954,18.817148,246.873542,326.862571,429.684953,877.605508
taxa_mortes_violentas_100k,162.0,33.504045,16.348876,7.728068,21.718322,29.805104,43.502108,81.281462
taxa_homicidios_100k,162.0,28.803584,14.710225,4.864326,17.497260,26.604705,38.193991,77.989689
taxa_feminicidios_100k,162.0,0.394042,0.422456,0.000000,0.000000,0.296187,0.668361,2.233931
taxa_estupros_100k,162.0,32.149043,17.599890,0.000000,20.651956,26.515021,43.172543,80.240087
taxa_furto_veiculos_100k,162.0,136.370873,92.989077,0.000000,67.732066,105.946956,195.061776,515.655235


## 9. Criar features temporais

In [10]:
df_base = df_base.sort_values(['codigo_municipio', 'ano']).copy()

df_base['lag_1_taxa_crimes_100k'] = df_base.groupby('codigo_municipio')['taxa_crimes_100k'].shift(1)
df_base['lag_1_taxa_mortes_violentas_100k'] = df_base.groupby('codigo_municipio')['taxa_mortes_violentas_100k'].shift(1)
df_base['variacao_taxa_crimes_100k_pct'] = df_base.groupby('codigo_municipio')['taxa_crimes_100k'].pct_change() * 100

df_base[[
    'codigo_municipio',
    'nome_municipio',
    'ano',
    'taxa_crimes_100k',
    'lag_1_taxa_crimes_100k',
    'variacao_taxa_crimes_100k_pct',
]].head(12)


,codigo_municipio,nome_municipio,ano,taxa_crimes_100k,lag_1_taxa_crimes_100k,variacao_taxa_crimes_100k_pct
0,1100205,Porto Velho,2016,777.499384,NaN,NaN
1,1100205,Porto Velho,2017,757.279762,777.499384,-2.600597
2,1100205,Porto Velho,2018,743.555245,757.279762,-1.812344
3,1100205,Porto Velho,2019,832.036620,743.555245,11.899772
4,1100205,Porto Velho,2020,86.770470,832.036620,-89.571316
5,1100205,Porto Velho,2021,83.067372,86.770470,-4.267694
6,1200401,Rio Branco,2016,90.247138,NaN,NaN
7,1200401,Rio Branco,2017,91.409993,90.247138,1.288522
8,1200401,Rio Branco,2018,72.041979,91.409993,-21.188071
9,1200401,Rio Branco,2019,397.231654,72.041979,451.389149


## 10. Criar indice simples de risco

In [11]:
componentes_risco = [
    'taxa_mortes_violentas_100k',
    'taxa_homicidios_100k',
    'taxa_feminicidios_100k',
    'taxa_estupros_100k',
]

for coluna in componentes_risco:
    minimo = df_base[coluna].min()
    maximo = df_base[coluna].max()
    if maximo == minimo:
        df_base[f'{coluna}_norm'] = 0
    else:
        df_base[f'{coluna}_norm'] = (df_base[coluna] - minimo) / (maximo - minimo)

df_base['risco_indice'] = df_base[[f'{coluna}_norm' for coluna in componentes_risco]].mean(axis=1)

df_base[['nome_municipio', 'ano', 'risco_indice']].sort_values('risco_indice', ascending=False).head(10)


,nome_municipio,ano,risco_indice
33,Macapa,2019,0.633614
10,Rio Branco,2020,0.582077
32,Macapa,2018,0.577623
26,Belem,2018,0.566396
31,Macapa,2017,0.551380
55,Fortaleza,2017,0.531212
142,Campo Grande,2020,0.527209
35,Macapa,2021,0.523585
134,Porto Alegre,2018,0.523433
7,Rio Branco,2017,0.521301


## 11. Selecionar colunas finais da base analitica

In [12]:
colunas_finais = [
    'codigo_municipio',
    'codigo_uf_ibge',
    'sigla_uf',
    'nome_municipio',
    'nome_municipio_padronizado',
    'ano',
    'populacao_total',
    'populacao_crescimento_pct',
    'ano_referencia_idhm',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'ideb',
    'fluxo',
    'aprendizado',
    'nota_mt',
    'nota_lp',
    'crimes_total_indicadores',
    'mortes_violentas_intencionais',
    'homicidios_dolosos',
    'feminicidios',
    'estupros',
    'furto_veiculos',
    'taxa_crimes_100k',
    'taxa_mortes_violentas_100k',
    'taxa_homicidios_100k',
    'taxa_feminicidios_100k',
    'taxa_estupros_100k',
    'taxa_furto_veiculos_100k',
    'lag_1_taxa_crimes_100k',
    'lag_1_taxa_mortes_violentas_100k',
    'variacao_taxa_crimes_100k_pct',
    'risco_indice',
]

df_base_analitica = df_base[colunas_finais].copy()

df_base_analitica.head()


,codigo_municipio,codigo_uf_ibge,sigla_uf,nome_municipio,nome_municipio_padronizado,ano,populacao_total,populacao_crescimento_pct,ano_referencia_idhm,idhm,...,taxa_crimes_100k,taxa_mortes_violentas_100k,taxa_homicidios_100k,taxa_feminicidios_100k,taxa_estupros_100k,taxa_furto_veiculos_100k,lag_1_taxa_crimes_100k,lag_1_taxa_mortes_violentas_100k,variacao_taxa_crimes_100k_pct,risco_indice
0,1100205,11,RO,Porto Velho,porto velho,2016,499293,NaN,2010,0.736,...,777.499384,38.254091,34.648994,0.000000,76.508183,373.928735,NaN,NaN,NaN,0.443955
1,1100205,11,RO,Porto Velho,porto velho,2017,509323,2.008841,2010,0.736,...,757.279762,31.021572,29.843537,0.000000,69.700367,365.190655,777.499384,38.254091,-2.600597,0.381733
2,1100205,11,RO,Porto Velho,porto velho,2018,519531,2.004229,2010,0.736,...,743.555245,30.797007,29.257157,0.769925,68.330860,358.015210,757.279762,31.021572,-1.812344,0.460860
3,1100205,11,RO,Porto Velho,porto velho,2019,529544,1.927315,2010,0.736,...,832.036620,22.472165,19.828381,0.188842,65.150394,375.983865,743.555245,30.797007,11.899772,0.325392
4,1100205,11,RO,Porto Velho,porto velho,2020,539354,1.852537,2010,0.736,...,86.770470,29.850525,26.142385,0.927035,52.840991,0.000000,832.036620,22.472165,-89.571316,0.416316


## 12. Validacoes finais

In [13]:
validar_chave(df_base_analitica, ['codigo_municipio', 'ano'], 'Base analitica municipio ano')
print('Anos:', sorted(df_base_analitica['ano'].unique()))
print('Municipios:', df_base_analitica['codigo_municipio'].nunique())
mostrar_nulos(df_base_analitica)


Base analitica municipio ano: 162 linhas, 34 colunas
Duplicatas na chave ['codigo_municipio', 'ano']: 0
Anos: [2016, 2017, 2018, 2019, 2020, 2021]
Municipios: 27


,qtd_nulos
nota_lp,81
nota_mt,81
aprendizado,81
fluxo,81
ideb,81
variacao_taxa_crimes_100k_pct,27
lag_1_taxa_mortes_violentas_100k,27
lag_1_taxa_crimes_100k,27
populacao_crescimento_pct,27
furto_veiculos,0


## 13. Salvar base analitica

In [14]:
df_base_analitica.to_csv(CAMINHO_BASE_ANALITICA, index=False)
CAMINHO_BASE_ANALITICA


PosixPath('../datasets/tratados/base_analitica_municipio_ano.csv')

# Resultado da etapa

Arquivo gerado:

`datasets/tratados/base_analitica_municipio_ano.csv`

Essa base sera usada por:
- notebook 04 de modelagem
- notebook 05 de carga no PostgreSQL / DW
- Data Mart e Metabase
